## Resizing PVCs

You requested 10 GiB, it's full, you want 20. If the StorageClass has `allowVolumeExpansion: true` and the CSI driver supports it, just edit the request:

```bash
kubectl patch pvc data -p '{"spec":{"resources":{"requests":{"storage":"20Gi"}}}}'
# or: kubectl edit pvc data
```

The controller resizes the underlying disk, then the kubelet expands the filesystem inside the pod — online for most modern filesystems, **no restart needed.**

Constraints:

- **Grow only, never shrink** — smaller is rejected by the API server.
- **Not every driver supports it** — `kind`'s `local-path` doesn't; cloud drivers usually do.
- **A `FileSystemResizePending` condition** shows on the PVC until the filesystem-side expansion completes (`kubectl describe pvc <name>`).

If `allowVolumeExpansion: false`, the API server **rejects** the patch. The fix is to edit the StorageClass (or use a different one) and recreate the PVC — which means data migration, not a one-liner. So set expansion up front.

This closes the storage module: you can now provision durable storage, bind it, choose its access mode and reclaim policy, wire it into StatefulSets, and grow it. On our map that's the full **Storage** box — **PV**, **PVC**, **StorageClass**, **CSI** — the durable half of the Resources band, resolving onto the Pods that mount it.